# The Same Agent, Raw vs PydanticAI

**What you will build:** the exact weather-and-calculator agent from notebook 0.3 — but in PydanticAI. Side by side, you'll see precisely what the framework does for you, so you adopt it knowing what it hides.

This is a *Rosetta Stone* notebook: same task, two languages.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/10_rosetta_raw_vs_pydanticai.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.5 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

```{note}
OpenRouter speaks the OpenAI Chat Completions format, so we use `OpenAIProvider` with OpenRouter's `base_url` — the same pattern you used with the raw SDK in Block 0. One consistent way to reach any model, across all three blocks.
```

## Reminder: what we wrote by hand in 0.3

To make that agent, we wrote all of this ourselves:

- a `while` loop with a turn limit,
- a JSON schema describing every tool,
- code to read `tool_calls`, parse the JSON `arguments`, dispatch to the right function,
- code to append the assistant message and each `tool` result back into `messages`.

That was the point — you should know it cold. Now watch it collapse.

::::{dropdown} 🔍 The raw version from 0.3, verbatim — keep it open in a split screen
:color: info

```python
def run_agent(user_message, system="You are a helpful assistant. Use tools when they help.", max_turns=10):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user_message}]
    total_tokens = 0
    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=0,
        )
        total_tokens += resp.usage.total_tokens
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = tc.function.arguments
            try:
                args = json.loads(args)
                result = TOOL_FUNCTIONS[tc.function.name](**args)
            except Exception as e:
                result = f"Error: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "Stopped: reached max turns."

tools = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "parameters": {"type": "object",
                       "properties": {"city": {"type": "string"}},
                       "required": ["city"]}}},
    # ... and the same again for calculator
]
```

Plus the tool functions themselves (which PydanticAI also needs — the *functions* never go away, only the plumbing around them).
::::

## The same agent in PydanticAI

Tools are just Python functions with a docstring; you register them with `@agent.tool_plain`. The loop, the schemas, and the message plumbing are gone.

In [ ]:
agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@agent.tool_plain
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    fake = {"bilbao": "18C, light rain", "madrid": "31C, sunny", "london": "14C, cloudy"}
    return fake.get(city.lower(), "20C, clear")

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}
def safe_eval(expr):                       # same helper as notebook 0.3 -- no eval(), so 9**9**9 can't hang the kernel
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError('unsupported')
    return ev(ast.parse(expr, mode='eval'))

@agent.tool_plain
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * (3 + 4)'."""
    try:
        return str(safe_eval(expression))
    except Exception:
        return "Error: only +, -, *, / and parentheses are supported."

result = await agent.run("What's the weather in Madrid, and what is 231 * 17?")
print(result.output)

Same answer as 0.3, a third of the code. Notice what you did **not** write:

| In 0.3 you wrote by hand | In PydanticAI |
|--------------------------|---------------|
| The `while` loop + turn limit | Handled by `agent.run()` (the limit is configurable — `UsageLimits`, 1.5b) |
| A JSON schema per tool | Generated from the type hints |
| Parsing `tool_calls` and JSON arguments | Automatic |
| Appending tool results to `messages` | Automatic |
| `str` in, `str` out | Typed results (next notebooks) |

```{note}
The framework didn't add magic — it removed boilerplate. Everything it does, you already did by hand in Block 0. That's why we built the raw version first: you can now debug PydanticAI, because you know what it's doing underneath.
```

Don't take the table on faith — check it. `result.all_messages()` is the very list you built by hand, typed:

In [ ]:
for m in result.all_messages():
    for p in m.parts:
        detail = getattr(p, "content", None) or getattr(p, "args", "")
        print(f"{m.kind:8s} | {p.part_kind:13s} | {str(detail)[:60]}")

Line it up with 0.3's `messages` list: `system-prompt` + `user-prompt` are your initial two dicts, each `tool-call` is an assistant message with `tool_calls`, each `tool-return` is a `role: "tool"` message, and the final `text` is the answer. **Same protocol, same list — now with types.** When something misbehaves, this printout is where you'll look (1.9 builds the habit).

## Why this framework, and not another

Worth making the criteria explicit, because the *framework* matters less than *why* you'd pick one:

- **Validation is the core, not a feature.** Agent outputs you can trust are typed outputs (0.2's whole lesson), and PydanticAI is built by the Pydantic team — the library that already defines validation for the Python ecosystem. Typed outputs, tool schemas from type hints, and retry-on-validation-error are the framework's spine, not add-ons.
- **Tools are plain functions.** No wrappers, no registries — a decorator on a normal function (you just saw it).
- **It stays close to the metal.** `all_messages()` above *is* your Block 0 list; nothing is hidden beyond reach. Frameworks that abstract the messages away are harder to debug precisely when you need debugging most.
- **The ecosystem follows the same philosophy** — `pydantic-evals` (1.6) and Logfire observability come from the same team and compose with the same types.

The honest alternatives: **OpenAI's Agents SDK** (clean, but organized around one vendor's primitives), **LangChain agents** (a much bigger ecosystem — you'll adopt its graph runtime, LangGraph, in Block 2 for exactly the things PydanticAI doesn't do), and **raw SDK + `instructor`** (typed extraction without an agent loop — fine until you need the loop). The criteria above transfer to evaluating any of them; 3.3 returns to this choice with the whole course behind it.

::::{dropdown} 🔍 What PydanticAI kept from the raw version
:color: info

It's the *same* protocol from 0.0, just automated:

```
you:   @agent.tool_plain def calculator(...)      -> PydanticAI builds the JSON schema
run:   await agent.run(question)                  -> PydanticAI runs the THINK/ACT/OBSERVE loop
          model asks for calculator(...)          ->   PydanticAI parses args, calls your function
          model asks for get_weather(...)         ->   ... appends results, loops
          model returns text                      ->   loop ends
result.output                                     -> the final answer
```
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a third tool.** Write `word_count(text: str) -> int` in both versions — the raw loop from 0.3 and this PydanticAI one — and compare how much boilerplate each addition costs. **Done when:** you can name every piece the decorator wrote for you.
2. **Swap the model.** Change `MODEL_NAME` to a Claude or GPT id and confirm the same code runs unchanged.
3. **Find the turn limit.** The table says `agent.run()` handles the `max_turns` guard you wrote by hand. Prove it exists: make the agent hit a limit on purpose. **Done when:** you've seen `UsageLimits` stop a run.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Third tool.** PydanticAI side, in full:

```python
@agent.tool_plain
def word_count(text: str) -> int:
    """Count the number of words in a text."""
    return len(text.split())
```

Raw side: the function, **plus** a schema dict (name, description, parameters, required), **plus** an entry in `TOOL_FUNCTIONS`. Three edits vs one — and the schema is the error-prone part, which is exactly what the type hints generate.

**2 — Swap the model:** one string. If a model errors on tool calls, that's the model's tool-calling support, not your code — same lesson as 0.3.

**3 — The turn limit:**

```python
from pydantic_ai.usage import UsageLimits

result = await agent.run(
    "What's the weather in Madrid, and what is 231 * 17?",
    usage_limits=UsageLimits(request_limit=1),      # one model call allowed — the tools need two
)
```

This raises `UsageLimitExceeded` — the typed cousin of your `max_turns` valve. The full toolkit (request limits, token limits, and testing without a model at all) is **1.5b**.
::::

**What's next:** in **1.1** we slow down and cover the `Agent` object properly — system prompts, running, and the result object.